# AI-improves-itself — notebook de recherche v2 (colab/)

Ce notebook exécute les **tâches de recherche programmées par l'IA** (page `/colab` du site) :
`search` (requête), `fetch` (URL), `deep` (recherche + lecture auto des meilleures pages), `note`.

| Cellule | Rôle |
|---|---|
| 1. Config | URL de ton site, limites, résumé IA local (optionnel, clé API) |
| 2. Diagnostic | télécharge `main.py`, réveille le serveur, affiche backend + file fusionnée |
| 3. Exécution | lance les tâches, pousse statuts + résultats au site |
| 4. Résultats | cartes lisibles + bilans d'étude serveur + lien vers `/colab` |

**Coût : 0 € sans clé** (recherche multi-moteurs sans clé). Avec clé : ~1 appel/tâche sur ton tier gratuit.
Astuce : **Runtime → Run all** enchaîne tout.

In [ ]:
# CONFIG — modifie puis exécute (ou Runtime → Run all pour tout enchaîner)
MAIN_SITE_URL = "https://aiis-core.onrender.com"  # URL de TON site, sans / final
BRANCH = "main"          # branche GitHub d'où vient main.py (nom d'une branche de dev pour tester une PR)
MAX_TASKS = 10           # tâches max par exécution (sessions Colab limitées)
FETCH_TOP = 1            # search : lit aussi le texte des N premiers résultats (0 = titres seuls)
SUMMARIZE = False        # True = résumé IA local de chaque tâche (nécessite une clé ci-dessous)
LLM_PROVIDER = "mistral" # mistral | gemini | groq | openrouter | nvidia
LLM_MODEL = ""           # vide = modèle économe par défaut du provider

import os
if SUMMARIZE:
    try:
        from getpass import getpass
        _key = getpass("Clé API (reste en RAM, jamais affichée ni sauvegardée) : ").strip()
    except Exception:
        _key = input("Clé API : ").strip()
    os.environ["COLAB_LLM_API_KEY"] = _key
    os.environ["COLAB_LLM_PROVIDER"] = LLM_PROVIDER
    os.environ["COLAB_LLM_MODEL"] = LLM_MODEL
    del _key
    print("Clé chargée en mémoire (non affichée, non sauvegardée, jamais poussée au site).")
else:
    os.environ.pop("COLAB_LLM_API_KEY", None)
    print("Sans résumé local (SUMMARIZE=False) — le site étudiera les résultats après le push.")

In [ ]:
# DIAGNOSTIC — récupère main.py, réveille le serveur, affiche la file fusionnée (sans exécuter)
import importlib
import os
import urllib.request

os.environ["MAIN_SITE_URL"] = MAIN_SITE_URL
os.environ["COLAB_BRANCH"] = BRANCH

# Seul main.py est téléchargé : tasks.json N'EST PLUS écrasé à chaque run
# (c'était le bug v1 : les tâches done ressuscitaient en pending).
REPO = "Snowoo-2z/AI-improves-itself"
url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/colab/main.py"
urllib.request.urlretrieve(url, "main.py")
print("↓ main.py depuis", BRANCH)

import main
importlib.reload(main)
assert getattr(main, "COLAB_SCRIPT_VERSION", 1) >= 2, (
    "main.py trop ancien (v1) : vérifie BRANCH, ou mets à jour le repo."
)
main.MAIN_SITE_URL = MAIN_SITE_URL

llm_cfg = main.llm_config_from_env() if SUMMARIZE else None
report = main.preflight(llm_cfg=llm_cfg, branch=BRANCH)
if not report["pending"]:
    print("\nAstuce : demande quelque chose à l'IA sur le site (elle programmera des tâches),"
          " ou ajoute-en une à la main sur la page /colab, puis ré-exécute.")

In [ ]:
# EXÉCUTION — lance les tâches pending, pousse au site, attend l'étude serveur
done = main.run_all(
    max_tasks=MAX_TASKS,
    fetch_top=FETCH_TOP,
    summarize=SUMMARIZE,
    llm=main.llm_config_from_env() if SUMMARIZE else None,
)
print(f"\n✅ {len(done)} tâche(s) exécutée(s).")

In [ ]:
# RÉSULTATS — cartes lisibles (extraits, résumés, bilans d'étude serveur)
main.show_results(limit=8, poll=True)
print(f"\n🔗 Page /colab du site : {MAIN_SITE_URL.rstrip('/')}/colab.html")

## Dépannage

| Symptôme | Cause probable | Solution |
|---|---|---|
| `serveur injoignable` | URL du site fausse, ou Render en redéploiement | vérifie `MAIN_SITE_URL` (sans `/` final), ré-exécute 2 min plus tard |
| `moteur : wikipedia-*` au lieu de `ddg-*` | DuckDuckGo a bloqué/renvoyé vide (fréquent depuis Colab) | normal : fallback automatique, le résultat reste valide |
| `aucun résultat (…)` | tous les moteurs ont échoué (réseau Colab filtré ?) | ré-exécute ; si ça persiste, reformule la tâche en `fetch` d'URL directe |
| `résumé impossible : 401` | clé API invalide | re-saisis la clé (cellule 1), vérifie le provider |
| `résumé impossible : 429` | quota gratuit atteint | le brut est quand même poussé ; réessaie demain ou change de provider |
| `étude(s) pas encore prête(s)` | le site met ~30-60 s (2 appels LLM) | recharge `/colab` dans 1 min — rien n'est perdu |
| `main.py trop ancien (v1)` | `BRANCH` pointe sur du vieux code | `BRANCH = "main"` après merge, ou nom de la branche de dev testée |